# Tutorial 9 — In Vitro–In Vivo Correlation (IVIVE)
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

IVIVE evaluates how well gene expression changes in cell cultures (in vitro) predict changes in whole animals (in vivo). This is central to my toxicology work at BHSAI — reducing animal use while improving translational relevance.

In [ ]:
!pip install pandas numpy matplotlib scipy seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(7)
n_genes = 500

# Simulate log2FC for a chemical (e.g., thioacetamide) in two systems
invivo_lfc  = np.random.normal(0, 2, n_genes)

# Good IVIVE: ~75% concordance with some scatter
invitro_lfc = invivo_lfc * 0.72 + np.random.normal(0, 0.9, n_genes)

# Add some genes that are system-specific (no IVIVE)
invitro_lfc[420:460] = np.random.normal(0, 2, 40)   # in-vitro specific
invivo_lfc[460:480]  = np.random.normal(0, 2, 20)   # in-vivo specific

genes = [f"Gene_{i:04d}" for i in range(n_genes)]
df = pd.DataFrame({"gene": genes, "invivo_lfc": invivo_lfc, "invitro_lfc": invitro_lfc})
df["concordant"] = np.sign(df["invivo_lfc"]) == np.sign(df["invitro_lfc"])
print(f"Concordance rate: {df['concordant'].mean():.1%}")

In [ ]:
r, pval = stats.pearsonr(df["invivo_lfc"], df["invitro_lfc"])
rho, _ = stats.spearmanr(df["invivo_lfc"], df["invitro_lfc"])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Scatter with regression
ax = axes[0]
ax.scatter(df["invivo_lfc"], df["invitro_lfc"], alpha=0.4, s=15, c="#1565c0")
m, b = np.polyfit(df["invivo_lfc"], df["invitro_lfc"], 1)
x_line = np.linspace(df["invivo_lfc"].min(), df["invivo_lfc"].max(), 100)
ax.plot(x_line, m*x_line+b, color="#e65100", lw=2)
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("In vivo log2FC (rat)"); ax.set_ylabel("In vitro log2FC (human hepatocytes)")
ax.set_title(f"IVIVE Correlation
Pearson r={r:.3f}, Spearman ρ={rho:.3f}")

# 2. Concordance per LFC bin
bins = pd.cut(df["invivo_lfc"].abs(), bins=[0,0.5,1,1.5,2,3,100],
              labels=["<0.5","0.5-1","1-1.5","1.5-2","2-3",">3"])
conc_by_bin = df.groupby(bins)["concordant"].mean()
axes[1].bar(range(len(conc_by_bin)), conc_by_bin.values, color="#00897b", edgecolor="white")
axes[1].set_xticks(range(len(conc_by_bin))); axes[1].set_xticklabels(conc_by_bin.index, rotation=30)
axes[1].axhline(0.5, color="red", linestyle="--", lw=0.8, label="Random")
axes[1].set_ylabel("Concordance rate"); axes[1].set_title("Concordance by |LFC| bin")
axes[1].legend()

# 3. Distribution of LFC difference
diff = df["invitro_lfc"] - df["invivo_lfc"]
axes[2].hist(diff, bins=40, color="#7b1fa2", edgecolor="white", alpha=0.8)
axes[2].axvline(0, color="red", lw=1.5)
axes[2].set_xlabel("Δ LFC (in vitro − in vivo)"); axes[2].set_ylabel("# genes")
axes[2].set_title(f"LFC difference distribution
μ={diff.mean():.3f}, σ={diff.std():.3f}")

plt.tight_layout(); plt.savefig("ivive.png", dpi=150); plt.show()

## Cross-species comparison

In [ ]:
# Simulated: rat vs guinea pig vs human for the same chemical
np.random.seed(12)
rat_lfc    = np.random.normal(0, 2, 200)
gp_lfc     = rat_lfc * 0.8 + np.random.normal(0, 0.8, 200)
human_lfc  = rat_lfc * 0.65 + np.random.normal(0, 1.1, 200)

pairs = [("Rat vs Guinea Pig", rat_lfc, gp_lfc),
         ("Rat vs Human",      rat_lfc, human_lfc),
         ("Guinea Pig vs Human", gp_lfc, human_lfc)]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (title, x, y) in zip(axes, pairs):
    r, _ = stats.pearsonr(x, y)
    ax.scatter(x, y, alpha=0.4, s=12, c="#1565c0")
    m, b = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    ax.plot(xl, m*xl+b, color="#e65100", lw=2)
    ax.set_title(f"{title}
r={r:.3f}"); ax.set_xlabel("Species 1 LFC"); ax.set_ylabel("Species 2 LFC")
plt.tight_layout(); plt.savefig("cross_species.png", dpi=150); plt.show()

## Key takeaways
- IVIVE r > 0.7 is considered good translational relevance
- High |LFC| genes show better concordance — small changes are dominated by noise
- Guinea pig often outperforms rat for hepatotoxicity IVIVE (as shown in my 2023 paper)
- See: Goel H et al. *Int. J. Mol. Sci.* 2023, 24, 7434